# Aula 27 - Arquivos de texto e CSV

In [4]:
import pprint as pp
import csv

In [5]:
%%writefile alunos.csv
matricula,nome,turma,nota
101,ana silva,a,8.5
102,bruno souza,b,7.0
103,carla dias,a,9.2
104,diego lima,b,6.4

Overwriting alunos.csv


In [6]:
%%writefile produtos.csv
codigo;nome;quantidade;preco
1;teclado;15;120.50
2;mouse;30;65.00
3;monitor;8;850.00
4;headset;12;210.90

Overwriting produtos.csv


In [7]:
%%writefile clientes.csv
id,nome_completo,cidade,limite
1,"albuquerque, carlos",fortaleza,2500.00
2,"mendes, beatriz",recife,4300.00
3,"santos, joao",salvador,1800.00

Overwriting clientes.csv


## 1. Persistência e ciclo de vida de arquivos

Até este ponto, qualquer variável, lista ou dicionário residia exclusivamente na memória de trabalho (RAM). A memória RAM é volátil: ao término do programa ou em caso de falta de energia, os dados são perdidos.

A memória secundária (disco rígido ou SSD) permite a persistência dos dados entre diferentes execuções.

### `open` e `close`

Para que o Python acesse o disco, ele precisa solicitar ao sistema operacional a abertura de um canal de dados denominado fluxo ou stream. Esse fluxo precisa ser requisitado, manipulado e encerrado de forma explícita.

In [8]:
# abrindo canal no modo de escrita ('w')
entrada = open("dados.txt", "w", encoding="utf-8")

entrada.write("primeira linha de dados\n")
entrada.write("segunda linha de dados\n")

# fechando o fluxo para transferir o buffer ao disco
entrada.close()

Durante a escrita, o sistema operacional não grava cada caractere imediatamente no disco para evitar lentidão. Ele acumula as informações em uma região temporária chamada buffer. A chamada do método `close()` garante o esvaziamento desse buffer no disco e devolve o descritor de arquivo ao sistema operacional.


### `read`

Para resgatar esses dados de volta para a memória principal, utilizamos o modo de leitura ('r'). O método `read()` transfere o conteúdo do fluxo diretamente para uma cadeia de caracteres na memória.

In [9]:
# abrindo canal no modo de leitura ('r')
entrada = open("dados.txt", "r", encoding="utf-8")

conteudo = entrada.read()
print(conteudo)

entrada.close()

primeira linha de dados
segunda linha de dados



In [10]:
# abrindo canal no modo de leitura ('r')
entrada = open("dados.txt", "r", encoding="utf-8")

conteudo = ""
while True:
    caractere = entrada.read(1)
    conteudo += caractere
    if caractere == "":
        break

print(conteudo)

entrada.close()

primeira linha de dados
segunda linha de dados



### `with`

O fechamento manual do arquivo envolve riscos. Se ocorrer um erro ou uma exceção antes da instrução close(), o arquivo continuará retido pelo processo no sistema operacional, o que pode corromper dados ou esgotar recursos.

Para resolver isso, a linguagem oferece a estrutura `with`. Ela atua como um gerenciador de contexto: ao final do bloco de código indentado, o interpretador garante a chamada do fechamento do arquivo, mesmo diante de falhas.

In [11]:
# escrita segura usando gerenciador de contexto
with open("dados.txt", "a", encoding="utf-8") as entrada:
    entrada.write("terceira linha de dados em modo append\n")

O modo 'a' (append) acima permite acrescentar novas informações ao final do arquivo existente sem apagar o conteúdo prévio.

## 2. Estratégias de leitura, ponteiro de fluxo e tratamento de exceções

A comunicação com o disco rígido é uma operação externa ao interpretador e sujeita a imprevistos que fogem do controle do algoritmo, como a ausência do arquivo solicitado, falta de permissão de acesso ou espaço insuficiente em disco. Por essa razão, operações de entrada e saída (I/O) devem ser integradas aos blocos de tratamento de exceções vistos na aula anterior.

Antes da introdução do gerenciador de contexto `with`, a liberação de recursos em caso de falha dependia exclusivamente do bloco `finally`. A estrutura `with` pode ser combinada diretamente com blocos `try`, `except` e `else` para tratar cenários em que o arquivo simplesmente não existe no caminho especificado (FileNotFoundError).

In [12]:
# tenta abrir e trata as exceções se falharem
nome_arquivo = "relatorio_inexistente.txt"

try:
    with open(nome_arquivo, "r", encoding="utf-8") as entrada:
        conteudo = entrada.read()

# ERROS DO OPEN()
except FileNotFoundError:
    print("O sistema não encontrou o arquivo informado.")

except PermissionError:
    print("O programa não tem permissão para ler este arquivo.")

except IsADirectoryError:
    print("O caminho informado aponta para uma pasta, não para um arquivo.")

# ERROS DO READ()
except UnicodeDecodeError as e:
    print(f"Erro de codificação: O arquivo não está em UTF-8 ou contém bytes inválidos. Detalhe: {e}")

except MemoryError:
    print("O arquivo é grande demais para ser carregado na memória RAM disponível.")

# ERRO GENÉRICO DE ENTRADA/SAÍDA
except OSError as e:
    # Captura outros erros de SO/Hardware (disco corrompido, cabo de rede desconectado, etc.)
    # obs.: FileNotFoundError e PermissionError também são OSErrors, mas foram tratados acima.
    print(f"Erro de sistema operacional ao manipular o arquivo: {e}")

# ERRO INESPERADO
except Exception as e:
    # Captura erros gerais de lógica ou bugs internos.
    # obs.: não captura Ctrl+C (KeyboardInterrupt) nem sys.exit() (SystemExit).
    print(f"Ocorreu um erro inesperado: {e}")

# QUALQUER OUTRA COISA (não recomendado)
except BaseException as e:
    print("Execução interrompida:", e)

else:
    print("Leitura concluída com sucesso.")

O sistema não encontrou o arquivo informado.


Carregar todo o conteúdo de um arquivo para uma string com o método `read()` só é viável quando o tamanho do dado cabe com folga na memória RAM. Em conjuntos de dados volumosos, carregar o arquivo de uma só vez pode esgotar a memória do computador.

O sistema operacional resolve esse problema permitindo a leitura sequencial sob demanda. O próprio objeto de arquivo em Python funciona como um iterador: ele carrega do disco para a memória apenas uma linha por vez durante a passagem do laço for.

In [13]:
# leitura sequencial linha por linha sem sobrecarregar a memoria
with open("dados.txt", "r", encoding="utf-8") as entrada:
    for linha in csv.reader(entrada, delimiter=" "):
        # cada linha lida traz consigo o caractere de nova linha (\n)
        print(linha)

['primeira', 'linha', 'de', 'dados']
['segunda', 'linha', 'de', 'dados']
['terceira', 'linha', 'de', 'dados', 'em', 'modo', 'append']


Internamente, o fluxo mantém um apontador de posição, frequentemente chamado de cursor ou offset de bytes. Toda operação de leitura avança esse cursor pelo fluxo de dados até encontrar o fim do arquivo (EOF - End Of File).

Se tentarmos ler o arquivo novamente usando o mesmo fluxo sem reposicionar o cursor, o Python retornará uma string vazia porque o apontador já alcançou o final dos dados. O método tell() revela a posição atual em bytes, e o método seek() permite mover o cursor de volta para qualquer ponto do arquivo.

In [14]:
# inspecionando e reposicionando o cursor interno
with open("dados.txt", "r", encoding="utf-8") as entrada:
    print("posicao inicial do cursor:", entrada.tell())

    primeira_leitura = entrada.read()
    print("posicao apos leitura completa:", entrada.tell())

    # reposicionando o cursor no inicio do arquivo (byte zero)
    entrada.seek(0)
    print("posicao apos o comando seek(0):", entrada.tell())

posicao inicial do cursor: 0
posicao apos leitura completa: 89
posicao apos o comando seek(0): 0


In [15]:
# relendo a primeira linha sem reposicionamento
with open("dados.txt", "r", encoding="utf-8") as entrada:
    primeira_leitura = entrada.read()
    print("posicao apos leitura completa:", entrada.tell())

    primeira_linha = entrada.readline().rstrip("\n")
    print("conteudo da primeira linha:", primeira_linha)

posicao apos leitura completa: 89
conteudo da primeira linha: 


In [16]:
# relendo a primeira linha com reposicionamento
with open("dados.txt", "r", encoding="utf-8") as entrada:
    primeira_leitura = entrada.read()
    print("posicao apos leitura completa:", entrada.tell())

    entrada.seek(0) # reposicionando

    primeira_linha = entrada.readline().rstrip("\n")
    print("conteudo da primeira linha:", primeira_linha)

posicao apos leitura completa: 89
conteudo da primeira linha: primeira linha de dados


## 3. Estrutura csv e processamento manual de delimitadores

O formato CSV (comma-separated values) nada mais é do que um arquivo de texto convencional estruturado sob uma convenção: cada linha representa um registro e os campos desse registro são separados por um caractere divisor, chamado de delimitador (frequentemente vírgula ou ponto e vírgula). A primeira linha costuma conter os nomes das colunas (cabeçalho).

Já há várias bibliotecas para leitura de CSVs, mas antes vamos compreender como poderíamos implementar manualmente com manipulação de strings.

In [17]:
# leitura manual do arquivo alunos.csv
with open("alunos.csv", "r", encoding="utf-8") as entrada:
    linhas = entrada.readlines()

pp.pprint(linhas)

['matricula,nome,turma,nota\n',
 '101,ana silva,a,8.5\n',
 '102,bruno souza,b,7.0\n',
 '103,carla dias,a,9.2\n',
 '104,diego lima,b,6.4\n']


In [18]:
# leitura manual do arquivo alunos.csv
lista_de_linhas = []
with open("alunos.csv", "r", encoding="utf-8") as entrada:
    while True:
        linha = entrada.readline()
        if linha == "":
            break
        lista_de_linhas.append(linha)

print(lista_de_linhas)

['matricula,nome,turma,nota\n', '101,ana silva,a,8.5\n', '102,bruno souza,b,7.0\n', '103,carla dias,a,9.2\n', '104,diego lima,b,6.4\n']


Todo dado obtido a partir de um arquivo chega à memória principal estritamente sob o tipo texto (string). Cabe ao algoritmo converter os campos numéricos para os seus tipos adequados (int ou float) caso operações aritméticas sejam necessárias.

Podemos separar a primeira linha para servir de referência para os campos e converter as linhas subsequentes em uma lista de dicionários na memória RAM.

In [19]:
# estruturando registros em memoria com tipagem manual
lista_alunos = []

with open("alunos.csv", "r", encoding="utf-8") as entrada:
    cabecalho = entrada.readline().rstrip("\n").split(",")

    for linha in entrada:
        linha = linha.strip()
        if linha == "":
            continue
        valores = linha.split(",")
        try:
            aluno = {
                cabecalho[0]: int(valores[0]),
                cabecalho[1]: valores[1],
                cabecalho[2]: valores[2],
                cabecalho[3]: float(valores[3])
            }
        except (ValueError, IndexError) as e:
            print(e)
            continue
        lista_alunos.append(aluno)

pp.pprint(lista_alunos)

[{'matricula': 101, 'nome': 'ana silva', 'nota': 8.5, 'turma': 'a'},
 {'matricula': 102, 'nome': 'bruno souza', 'nota': 7.0, 'turma': 'b'},
 {'matricula': 103, 'nome': 'carla dias', 'nota': 9.2, 'turma': 'a'},
 {'matricula': 104, 'nome': 'diego lima', 'nota': 6.4, 'turma': 'b'}]


In [20]:
# calculo de media aritmetica sobre os dados processados
soma_notas = 0.0

for aluno in lista_alunos:
    soma_notas += aluno["nota"]

media_turma = soma_notas / len(lista_alunos)
print("media geral da turma:", round(media_turma, 2))

media geral da turma: 7.78


A abordagem manual de separar registros usando `split(",")` falha silenciosamente quando os próprios dados contêm vírgulas. No padrão CSV, campos que possuem o caractere delimitador ou quebras de linha internas precisam ser envolvidos por aspas.

Ao aplicar `split(",")` de forma ingênua sobre o arquivo clientes.csv, a coluna de nome ("albuquerque, carlos") é dividida ao meio, gerando mais campos do que as colunas do cabeçalho e corrompendo a leitura dos registros.

In [21]:
# demonstracao de quebra de estrutura com split ingênuo
with open("clientes.csv", "r", encoding="utf-8") as entrada:
    cabecalho = entrada.readline().rstrip("\n").split(",")
    print("quantidade esperada de colunas:", len(cabecalho))
    print()

    for linha in entrada:
        l = linha.rstrip("\n")
        print(l)
        partes = l.split(",")
        print("campos encontrados:", len(partes), "->", partes)
        print()

quantidade esperada de colunas: 4

1,"albuquerque, carlos",fortaleza,2500.00
campos encontrados: 5 -> ['1', '"albuquerque', ' carlos"', 'fortaleza', '2500.00']

2,"mendes, beatriz",recife,4300.00
campos encontrados: 5 -> ['2', '"mendes', ' beatriz"', 'recife', '4300.00']

3,"santos, joao",salvador,1800.00
campos encontrados: 5 -> ['3', '"santos', ' joao"', 'salvador', '1800.00']



## 4. O módulo padrão csv

Para lidar de forma correta com textos entre aspas, vírgulas internas e variações de delimitadores, a biblioteca padrão do Python disponibiliza o módulo `csv`. Ele implementa internamente um analisador léxico conforme a especificação técnica RFC 4180.

A função `csv.reader()` lê o fluxo de texto e trata as aspas automaticamente, mantendo campos compostos íntegros em listas de strings.

In [22]:
import csv # IMPORTAÇÃO DE MÓDULO. A partir desta linha, "csv" permite acessar funções do módulo

# leitura do arquivo clientes.csv tratando aspas e virgulas internas
with open("clientes.csv", "r", encoding="utf-8") as entrada:
    leitor = csv.reader(entrada, delimiter=",")
    cabecalho = next(leitor)
    for linha in leitor:
        print(linha)

['1', 'albuquerque, carlos', 'fortaleza', '2500.00']
['2', 'mendes, beatriz', 'recife', '4300.00']
['3', 'santos, joao', 'salvador', '1800.00']


Muitas fontes de dados utilizam outros caracteres como separador de colunas, como o ponto e vírgula, muito comum em arquivos exportados em regiões onde a vírgula é o separador decimal.


### Leitura estruturada

Para simplificar a associação das colunas aos seus significados, a classe `csv.DictReader` consome a primeira linha como cabeçalho e entrega cada linha subsequente mapeada diretamente em um dicionário. O parâmetro `delimiter` permite especificar o caractere divisor utilizado no arquivo. O atributo `fieldnames` do leitor é a lista de campos.

In [23]:
%%writefile produtos.csv
codigo;nome;quantidade;preco
1;teclado;15;120.50
2;mouse;30;65.00
3;monitor;8;850.00
4;headset;12;210.90

# leitura de produtos com delimitador customizado usando DictReader
with open("produtos.csv", "r", encoding="utf-8") as arquivo:
    leitor_dict = csv.DictReader(arquivo, delimiter=";")
    print('CAMPOS', leitor_dict.fieldnames)
    for registro in leitor_dict:
        nome_produto = registro["nome"]
        qtd_estoque = int(registro["quantidade"])
        preco_unitario = float(registro["preco"])
        print(nome_produto, "-> total em itens:", qtd_estoque)

Overwriting produtos.csv


### Escrita estruturada

Para gravar dados estruturados de volta ao disco, usamos o construtor `csv.writer()` ou `csv.DictWriter()`.

Ao abrir um arquivo para gravação com o módulo `csv`, é fundamental passar o parâmetro `newline=""` na função `open()`. O módulo `csv` gerencia suas próprias quebras de linha de acordo com o padrão do formato; se esse parâmetro for omitido, o sistema operacional pode traduzir as quebras de linha duas vezes, gerando linhas em branco adicionais indesejadas no arquivo final.

In [24]:
# processamento e gravacao de dados derivados em um novo arquivo
with open("produtos.csv", "r", encoding="utf-8") as arquivo_entrada:
    leitor = csv.DictReader(arquivo_entrada, delimiter=";")

    with open("relatorio_estoque.csv", "w", encoding="utf-8", newline="") as arquivo_saida:
        escritor = csv.writer(arquivo_saida, delimiter=",")

        # gravacao manual da linha de cabecalho
        escritor.writerow(["nome_produto", "valor_patrimonial"])

        for item in leitor:
            quantidade = int(item["quantidade"])
            preco = float(item["preco"])
            valor_total = round(quantidade * preco, 2)

            # gravacao do registro processado no disco
            escritor.writerow([item["nome"], valor_total])

with open("relatorio_estoque.csv", "r", encoding="utf-8") as entrada:
    for linha in entrada:
        print(linha.rstrip("\n"))

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'

Quando os dados já se encontram organizados na memória como uma lista de dicionários, o objeto `csv.DictWriter` oferece uma abordagem mais declarativa. O método `writeheader()` descarrega os nomes das colunas e `writerows()` persiste múltiplos registros de uma só vez.

In [ ]:
# gravacao direta de colecao de dicionarios com DictWriter
novos_clientes = [
    {"id": 4, "nome_completo": "barros, mariana", "cidade": "natal", "limite": 3200.0},
    {"id": 5, "nome_completo": "fonseca, pedro", "cidade": "teresina", "limite": 1500.0}
]

campos = ["id", "nome_completo", "cidade", "limite"]

with open("novos_clientes.csv", "w", encoding="utf-8", newline="") as arquivo_saida:
    escritor_dict = csv.DictWriter(arquivo_saida, fieldnames=campos)

    escritor_dict.writeheader()
    escritor_dict.writerows(novos_clientes)

with open("novos_clientes.csv", "r", encoding="utf-8") as entrada:
    for linha in entrada:
        print(linha.rstrip("\n"))

id,nome_completo,cidade,limite
4,"barros, mariana",natal,3200.0
5,"fonseca, pedro",teresina,1500.0


## 5. Codificação de caracteres

O armazenamento em disco guarda exclusivamente sequências de bytes (zeros e uns). A interpretação desses bytes em caracteres de texto legíveis depende de uma tabela de codificação (encoding). Se um arquivo for gravado com uma codificação e lido com outra, caracteres acentuados serão corrompidos ou o interpretador disparará uma exceção UnicodeDecodeError.

O padrão universal moderno é o UTF-8. No entanto, sistemas corporativos legados e softwares como o Excel em ambientes Windows no Brasil frequentemente exportam arquivos no padrão ISO-8859-1 (também conhecido como Latin-1) ou Windows-1252. O parâmetro encoding na função open() é o mecanismo que informa ao interpretador qual tabela de tradução deve ser empregada.

In [ ]:
# simulando a geracao de um arquivo legado em formato latin-1
texto_com_acentos = "código;descrição;preço\n10;café moído;14.80\n"

with open("legado_latin1.csv", "w", encoding="latin-1") as entrada:
    entrada.write(texto_com_acentos)

In [ ]:
# tentando ler com utf-8 para observar a incompatibilidade de codificacao
try:
    with open("legado_latin1.csv", "r", encoding="utf-8") as entrada:
        conteudo = entrada.read()
except UnicodeDecodeError:
    print("falha na decodificacao: os bytes nao correspondem a tabela utf-8")

falha na decodificacao: os bytes nao correspondem a tabela utf-8


In [ ]:
# leitura bem-sucedida indicando explicitamente a codificacao original
with open("legado_latin1.csv", "r", encoding="latin-1") as entrada:
    conteudo = entrada.read()
    print(conteudo)

código;descrição;preço
10;café moído;14.80



Diferente da memória RAM, onde é possível inserir elementos no meio de uma lista com facilidade, a gravação em disco é estritamente sequencial. Se tentarmos escrever bytes no meio de um arquivo existente, o sistema operacional simplesmente sobrescreverá os dados que já estavam gravados naquela posição do disco, corrompendo a estrutura.

Por esse motivo, o padrão para atualizar, filtrar ou transformar arquivos em lote consiste em abrir o fluxo de entrada para leitura e, simultaneamente, abrir um novo fluxo de saída para escrita. O algoritmo consome o dado de origem linha a linha, aplica a transformação necessária e grava o resultado no destino.

In [ ]:
import csv

# padrao de transformacao por fluxo: atualizando registros com nova coluna calculada
with open("alunos.csv", "r", encoding="utf-8") as arq_origem:
    leitor = csv.DictReader(arq_origem)

    # definicao do novo conjunto de colunas incluindo o campo calculado
    campos_atualizados = leitor.fieldnames + ["situacao"]

    with open("alunos_processados.csv", "w", encoding="utf-8", newline="") as arq_destino:
        escritor = csv.DictWriter(arq_destino, fieldnames=campos_atualizados)
        escritor.writeheader()

        for linha in leitor:
            nota = float(linha["nota"])

            # transformacao condicional baseada na nota do registro
            if nota >= 7.0:
                situacao = "aprovado"
            else:
                situacao = "reprovado"

            linha["situacao"] = situacao
            escritor.writerow(linha)

with open("alunos_processados.csv", "r", encoding="utf-8") as entrada:
    for linha in entrada:
        print(linha.rstrip("\n"))

matricula,nome,turma,nota,situacao
101,ana silva,a,8.5,aprovado
102,bruno souza,b,7.0,aprovado
103,carla dias,a,9.2,aprovado
104,diego lima,b,6.4,reprovado


## Exercícios


### Exercício 1: log

Sistemas computacionais gravam continuamente o histórico de suas operações em arquivos de log de texto puro. Inspecione um arquivo de log linha por linha, capture com segurança apenas as ocorrências com nível de severidade ERROR e persista esses registros em um novo arquivo denominado erros_criticos.txt.

In [ ]:
%%writefile output_exercicios/servidor.log
2026-03-30 10:14:20 INFO conexao estabelecida pelo cliente 42
2026-03-30 10:14:22 WARNING tempo de resposta acima do esperado
2026-03-30 10:15:01 ERROR falha na conexao com o banco de dados
2026-03-30 10:15:10 INFO tentativa de reconexao automatica
2026-03-30 10:15:15 ERROR timeout ao requisitar recurso remoto

Writing output_exercicios/servidor.log


In [ ]:
with open("output_exercicios/servidor.log", "r", encoding="utf-8") as entrada:
    with open("output_exercicios/erros_criticos.txt", "w", encoding="utf-8") as saida:
        leitor = csv.reader(entrada, delimiter=" ")
        escritor = csv.writer(saida, delimiter=" ")
        for linha in leitor:
            if 'ERROR' not in linha:
                continue
            escritor.writerow(linha)
        


In [ ]:
# Confira aqui o arquivo erros_criticos.txt gerado pelo seu código
with open("output_exercicios/erros_criticos.txt", "r", encoding="utf-8") as entrada:
    print(entrada.read().rstrip("\n"))

2026-03-30 10:15:01 ERROR falha na conexao com o banco de dados

2026-03-30 10:15:15 ERROR timeout ao requisitar recurso remoto


### Exercício 2: Agrupamento no csv

Muitos relatórios financeiros usam o ponto e vírgula como separador. Leia uma tabela de vendas, agrupe os totais faturados por setor comercial usando estruturas de dicionário na memória RAM e persista o resumo calculado em um arquivo resumo_setores.csv delimitado por vírgula.

In [ ]:
%%writefile vendas.csv
vendedor;setor;valor
marcos;tecnologia;1400.00
aline;saude;850.50
renata;tecnologia;2100.00
fabio;educacao;620.00
lucas;saude;1250.00

Writing vendas.csv


In [ ]:
# Escreva sua solução aqui
with open('vendas.csv', 'r', encoding='utf-8') as entrada:
    resumo = {}

    leitor = csv.DictReader(entrada, delimiter=";")
    for linha in leitor:
        setor = linha["setor"]
        valor = float(linha["valor"])
        if setor not in resumo:
            resumo[setor] = valor
        else:
            resumo[setor] += valor
    print(resumo)

    linhas = []
    for s,t in resumo.items():
        linhas.append({'setor': s, 'total_faturado':t})

    with open('output_exercicios/resumo_setores.csv', 'w', encoding='utf-8', newline="") as saida:
        escritor = csv.DictWriter(saida, fieldnames=['setor', 'total_faturado'], delimiter=',')
        escritor.writeheader()
        escritor.writerows(linhas)

{'tecnologia': 3500.0, 'saude': 2100.5, 'educacao': 620.0}


In [ ]:
# Confira aqui o arquivo resumo_setores.csv gerado pelo seu código
with open("output_exercicios/resumo_setores.csv", "r", encoding="utf-8") as entrada:
    print(entrada.read().rstrip("\n"))

setor,total_faturado
tecnologia,3500.0
saude,2100.5
educacao,620.0


### Exercício 3: Limpeza de csv

Bases de dados reais frequentemente chegam com registros incompletos, textos vazios onde deveriam existir números ou tipos corrompidos. Filtre uma lista de estoque bruto, descarte linhas inválidas por meio de tratamento de exceções (ValueError) e grave apenas os itens válidos e saneados em estoque_limpo.csv.

In [ ]:
%%writefile output_exercicios/estoque_bruto.csv
codigo,item,quantidade,preco
1,teclado,10,120.00
2,mouse,invalido,45.00 #erro de conversão em qtd
3,fone,,80.00 #erro em qtd
4,webcam,5,190.50
5,gabinete,8,preco_desconhecido #erro em preco

Writing output_exercicios/estoque_bruto.csv


In [34]:
# Escreva sua solução aqui
with open('output_exercicios/estoque_bruto.csv', 'r', encoding='utf-8') as entrada:
    leitor = csv.DictReader(entrada)
    with open('output_exercicios/estoque_limpo.csv', 'w', encoding='utf-8') as saida:
        escritor = csv.DictWriter(saida, fieldnames=['codigo', 'item', 'quantidade', 'preco'])
        escritor.writeheader()

        for linha in leitor:
            try:
                codigo = int(linha['codigo'])
                item = linha['item']
                quantidade = int(linha['quantidade'])
                preco = float(linha['preco'])
            except ValueError:
                continue
            except Exception as e:
                print(f'Erro inesperado: {e}')
            else:
                escritor.writerow(linha)


In [35]:
# Confira aqui o arquivo estoque_limpo.csv gerado pelo seu código
with open("output_exercicios/estoque_limpo.csv", "r", encoding="utf-8") as entrada:
    for linha in entrada:
        print(linha.rstrip("\n"))

codigo,item,quantidade,preco

1,teclado,10,120.00

4,webcam,5,190.50

